![](Autonomous Ecom Agent Workflow.png)

####Autonomous Customer Support Agentic Resolution Engine

**1. Major AI Concepts Used:**
Agentic AI (Single Agent): Autonomous control over specific tools to achieve the goal. It Reasons & Act to solve the user's problem before generating the final response.

Tool Calling (Function Calling): The LLM is provided with a JSON schema defined  functions (get_order_details, check_inventory, etc.).

ReAct Loop (Reason + Act) Think -> Decide -> Act -> Observe -> Finish: The agent Reasons about the current state, decides to Act (call a tool), Observes the output of that tool, and then repeats the process until it has enough information to provide a final answer.

Medallion Architecture:
Bronze: Raw ingestion of data (ecommerce_raw_reviews).
Gold: Highly refined, actionable data resulting from the AI processing (ecommerce_agent_response).

2. LLM Model: Meta Llama 3.3

Model Size: 70 Billion Parameters (70B) for complex reasoning and reliable tool calling capabilities.

Variant: Instruct (Instruction tuned to follow directions and act as an assistant).

Hosting Platform: Databricks Model Serving (via Foundation Model APIs). The model is hosted securely within the Databricks environment, accessed via an OpenAI-compatible client.

3. Types of Prompting Techniques Used
The system_prompt

Role Prompting (Persona): `"You are an autonomous customer support agent."*

Dynamic prompt Injection (Grounding): "...The customer is writing about Order ID: {order_id}."

Instruction Prompting for Tool Use: "Use tools to investigate issues before responding. Never guess order details."

Guardrail / Negative Prompting: "Do not give cash refunds."

Conditional Logic Prompting: "If an item is out of stock, issue store credit automatically and give the customer the code."

In [0]:
# ==============================================================================
# 1. SETUP THE DATABASES
# ==============================================================================
print("1. Creating and loading the Delta Tables...")
# (Tables setup remains the same as your previous working version)

# Insert sample customer reviews with their associated order IDs
spark.sql("CREATE OR REPLACE TABLE lakehousecat.default.ecommerce_raw_reviews (review_id STRING, order_id STRING, customer_review STRING)")
spark.sql("""INSERT INTO lakehousecat.default.ecommerce_raw_reviews VALUES 
  ('REV-001', 'ORD-111', 'The shoes are beautiful, but they ripped after two days of wearing them! I want my money back.'),
  ('REV-002', 'ORD-222', 'Shipping took 3 weeks. Absolutely unacceptable.'),
  ('REV-003', 'ORD-333', 'Perfect fit! Will definitely be buying from you guys again.'),
  ('REV-004', 'ORD-555', 'I received the wrong color. I ordered black but got blue. How do I fix this?')""")

# Create the orders table to act as our internal e-commerce database
spark.sql("CREATE OR REPLACE TABLE lakehousecat.default.ecommerce_orders (order_id STRING, customer_id STRING, item STRING, ordered_color STRING, shipped_color STRING, price DOUBLE)")
spark.sql("""INSERT INTO lakehousecat.default.ecommerce_orders VALUES 
  ('ORD-111', 'CUST-10', 'Dress Shoes', 'Brown', 'Brown', 90.0),
  ('ORD-222', 'CUST-45', 'Sneakers', 'White', 'White', 60.0),
  ('ORD-333', 'CUST-88', 'Boots', 'Black', 'Black', 150.0),
  ('ORD-555', 'CUST-99', 'Running Shoes', 'Black', 'Blue', 120.0)""")
display(spark.read.table("lakehousecat.default.ecommerce_raw_reviews"))
display(spark.read.table("lakehousecat.default.ecommerce_orders"))

In [0]:
import json
import os
from openai import OpenAI
from pyspark.sql import Row
# ==============================================================================
# 2. DEFINE THE AGENT'S TOOLS (Updated with required Justification)
# ==============================================================================
print("2. Giving AI - the Hands and tools ...")
# Define a tool to fetch order details directly from the table
def get_order_details(order_id, **kwargs):
    print(f"[Tool1] Executing Database Lookup for Order: {order_id}")
    res = spark.sql(f"SELECT * FROM lakehousecat.default.ecommerce_orders WHERE order_id = '{order_id}'").first()
    return json.dumps(res.asDict()) if res else json.dumps({"error": "Order not found"})
# Define a tool to check warehouse inventory
def check_inventory(item_name, color, **kwargs):
    print(f"[Tool2] Checking Warehouse for {item_name} in {color}")
    return json.dumps({"item": item_name, "color": color, "in_stock": False})
# Define a tool to simulate issuing store credit to a customer
def issue_store_credit(customer_id, amount, **kwargs):
    print(f"[Tool3] Issuing ${amount} store credit to Customer: {customer_id}")
    return json.dumps({"status": "Success", "credit_code": "CREDIT-XYZ-123", "amount": amount})

# Create the JSON schema that strictly defines these tools for the LLM to understand
#type : Specifies the type of structure ("function" or "AgenticTool").
#name: A unique, descriptive name for the tool (e.g., "get_weather" or "ScientificTextSummarizer").
#description: A clear, human-readable description of what the tool does. The LLM uses this description to decide when to use the tool.
#input_schema (or parameters): A JSON Schema object that strictly defines the expected input arguments for the tool.
#type : Must be "object".
#properties: An object where each key is an argument name, and its value defines the data type (e.g., "string", "number") and a description.
#required: A list of strings specifying which properties are mandatory for the tool to function correctly.

agent_tools = [
    {
        "type": "function",
        "function": {
            "name": "get_order_details",
            "description": "Look up an order by its ID.",
            "parameters": {
                "type": "object",
                "properties": {
                    "order_id": {"type": "string"},
                    "justification": {"type": "string", "description": "Explain exactly why you need to call this tool."}
                },
                "required": ["order_id", "justification"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "check_inventory",
            "description": "Check if an item/color is in stock. MUST call this before issuing credit for wrong/damaged items.",
            "parameters": {
                "type": "object",
                "properties": {
                    "item_name": {"type": "string"},
                    "color": {"type": "string"},
                    "justification": {"type": "string", "description": "Explain exactly why you need to call this tool."}
                },
                "required": ["item_name", "color", "justification"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "issue_store_credit",
            "description": "Issue store credit if replacement is out of stock.",
            "parameters": {
                "type": "object",
                "properties": {
                    "customer_id": {"type": "string"},
                    "amount": {"type": "number"},
                    "justification": {"type": "string", "description": "Explain exactly why you need to call this tool."}
                },
                "required": ["customer_id", "amount", "justification"]
            }
        }
    }
]

# Map the string names from the JSON schema to the actual Python functions
available_functions = {"get_order_details": get_order_details, "check_inventory": check_inventory, "issue_store_credit": issue_store_credit}

# ==============================================================================
# 3. CONFIGURE THE LLM CLIENT & AGENT LOGIC (Updated Prompting)
# ==============================================================================
DATABRICKS_TOKEN = dbutils.secrets.get(scope="izgenaiscope", key="databricks_token")

def run_ecommerce_agent(customer_review, order_id, api_token):
    client = OpenAI(api_key=api_token, base_url="https://3631047205072094.ai-gateway.cloud.databricks.com/mlflow/v1/")
    
    #Prompt Engineering: Strict instructions to force the detailed reasoning behavior
    system_prompt = f"""You are an autonomous customer support agent for Order ID: {order_id}.
    1. ALWAYS look up the order details first.
    2. If an item is damaged or the wrong color, you MUST check inventory for a replacement.
    3. If and ONLY IF the inventory is out of stock, issue store credit.
    4. You must provide a clear, professional justification for every tool call."""

    messages = [{"role": "system", "content": system_prompt},
        {"role": "user", "content": customer_review}]

# Reasoning Loop: The agent is given up to 5 turns to solve the problem. This prevents "infinite loops".
#Step A (Think): We send the prompt and the tools to the Databricks-hosted Llama 3.3 70B model.
#Step B (Decide): The AI responds. If it needs data, it doesn't return text; it returns a tool_call requesting to run a specific function.
#Step C (Act): If a tool was requested, your Python code extracts the tool name and arguments, executes the corresponding function from available_functions, and gets the JSON result (e.g., the order details from the database).
#Step D (Observe): The JSON result is appended to the messages history, and the loop starts over. The AI reads the database result and "thinks" about what to do next.
#Step E (Finish): If the AI has all the data it needs and resolves the issue, it outputs a standard text response. The loop breaks, and the final email draft is returned.
    for i in range(5):
        #LLM Inference & Tool Choice
        #Step A (Think): tool_calls request
        response = client.chat.completions.create(model="databricks-meta-llama-3-3-70b-instruct", messages=messages, tools=agent_tools)#Step B (Decide): Tool Call
        
        # message response 
        msg = response.choices[0].message
        #Step D (Observe): append the iteration of messages
        messages.append(msg.model_dump(exclude_none=True))
        
        ##Step E (Finish): No Tools Called - Agent reached conclusion, produce final response
        if not msg.tool_calls:
            print(f"\n FINAL AGENT RESPONSE:\n{msg.content}")
            return msg.content
        
        #Step C (Act):
        for tool in msg.tool_calls:
            #Extraction of tool arguments
            args = json.loads(tool.function.arguments)
            #Justification (if nothing then default no justification)
            print(f" AGENT JUSTIFICATION just for our understanding of the agent's reasoning: {args.get('justification', 'No justification provided.')}")
            #looks up the function object in the dictionary defined earlier
            result = available_functions[tool.function.name](**args)
            #Step D (Observe): append the iteration of messages
            #Tool call id - shows the tool request this result belongs to
            messages.append({"role": "tool", "tool_call_id": tool.id, "name": tool.function.name, "content": result})

# ==============================================================================
# 4. EXECUTE THE PIPELINE
# ==============================================================================
print("\n2. Reading raw data and starting Agentic Pipeline...")

reviews_list = spark.read.table("lakehousecat.default.ecommerce_raw_reviews").collect()
processed_records = []

for row in reviews_list:
    print("\n" + "="*60)
    print(f"NEW TICKET: {row['customer_review']}")
    print(f"ATTACHED ORDER ID: {row['order_id']}")
    print("-" * 60)
    
    reply = run_ecommerce_agent(row['customer_review'], row['order_id'], DATABRICKS_TOKEN)
    processed_records.append({"review_id": row['review_id'], "order_id": row['order_id'], "agent_response": reply})

df_gold = spark.createDataFrame([Row(**r) for r in processed_records])
display(df_gold)

![](agent flow.jpg)